# 06 — Entrenamiento TimeGAN

**Fase 2 — Modelo generativo para series temporales clínicas**  
Referencia: Yoon et al., *Time-series Generative Adversarial Networks* (NeurIPS 2019).

A diferencia de CTGAN/TVAE/TabDDPM (que operan sobre snapshots tabulares),
TimeGAN genera series temporales completas de 48h × 10 vitales.
Aprende tanto la distribución marginal como la dinámica temporal mediante
un espacio latente supervisado por una red Supervisor.

**Advertencia sobre GCS**: `gcs_verbal` tiene 39.8% de ceros y `gcs_eye` un 15.5%
(pacientes intubados/sedados). Esta distribución con masa puntual en 0 es un
caso difícil para los modelos generativos; se monitoriza explícitamente en la validación.

Fases de entrenamiento:
1. Pre-entrenamiento del autoencoder (Embedder + Recovery)
2. Pre-entrenamiento del Supervisor
3. Entrenamiento conjunto adversarial (las 5 redes)
4. Generación y validación

## 0. Imports y configuración

In [ ]:
import sys, warnings, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

ROOT      = Path("..")
PROCESSED = ROOT / "data" / "processed"
SYNTHETIC = ROOT / "data" / "synthetic"
MODELS    = ROOT / "models"
REPORTS   = ROOT / "reports"
for d in [SYNTHETIC, MODELS, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "src"))
from models.timegan import (
    Embedder, Recovery, Generator, Supervisor, Discriminator,
    reconstruction_loss, supervised_step_loss,
    generator_adversarial_loss, discriminator_loss, moments_loss,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
np.random.seed(42)
sns.set_theme(style="whitegrid", palette="muted")

## 1. Carga de datos

In [ ]:
ts   = np.load(PROCESSED / "timeseries_48h.npy").astype(np.float32)
meta = pd.read_parquet(PROCESSED / "timeseries_48h_meta.parquet")
norm = pd.read_csv(PROCESSED / "timeseries_norm_params.csv")

N, SEQ_LEN, N_FEATURES = ts.shape
VITAL_NAMES = norm["vital"].tolist()

print(f"Array timeseries: {ts.shape}  (estancias × horas × vitales)")
print(f"Vitales: {VITAL_NAMES}")
print(f"Rango: [{ts.min():.3f}, {ts.max():.3f}]  (ya normalizado [0,1] desde Fase 1)")
print(f"\nMortalidad: {meta['hospital_expire_flag'].mean()*100:.1f}%")

# Nota sobre GCS
gcs_cols = [i for i, v in enumerate(VITAL_NAMES) if "gcs" in v]
print("\nPorcentaje de ceros por canal GCS (distribución con masa puntual):")
for i in gcs_cols:
    pct = (ts[:, :, i] == 0).mean() * 100
    print(f"  {VITAL_NAMES[i]:<15} {pct:.1f}%")

## 2. Instanciación de modelos

### Justificación de hiperparámetros

| Parámetro | Valor | Justificación |
|---|---|---|
| `hidden_dim` | 24 | 2.4 × n_features (10); mismo orden que Yoon et al. para datasets de tamaño similar |
| `num_layers` | 3 | Captura dependencias temporales a corto, medio y largo plazo en las 48h |
| `noise_dim` | 10 | Igual a n_features; el generador parte de un ruido de la misma dimensionalidad que la señal |
| `batch_size` | 128 | Balance entre calidad de gradientes y memoria GPU para secuencias de 48 pasos |
| `pretrain_epochs` | 200 | Pre-entrenamiento hasta convergencia antes del entrenamiento adversarial |
| `joint_epochs` | 1000 | Entrenamiento conjunto; más épocas que los modelos tabulares por la complejidad temporal |
| `gamma` | 1 | Peso del moments loss; igual al paper original |
| `eta` | 10 | Peso del supervised loss en el joint training; refuerza la preservación de dinámica temporal |
| `d_threshold` | 0.15 | El discriminador solo se actualiza si su loss supera este umbral, evitando que domine al generador |

In [ ]:
HIDDEN_DIM    = 24
NUM_LAYERS    = 3
NOISE_DIM     = N_FEATURES
BATCH         = 128
PRETRAIN_E    = 200
PRETRAIN_S    = 200
JOINT_EPOCHS  = 1000
GAMMA         = 1.0    # peso moments loss
ETA           = 10.0   # peso supervised loss
D_THRESHOLD   = 0.15   # umbral para actualizar el discriminador

embedder     = Embedder(N_FEATURES, HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
recovery     = Recovery(HIDDEN_DIM, N_FEATURES, NUM_LAYERS).to(DEVICE)
generator    = Generator(NOISE_DIM, HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
supervisor   = Supervisor(HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
discriminator = Discriminator(HIDDEN_DIM, NUM_LAYERS).to(DEVICE)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total_params = sum(count_params(m) for m in
                   [embedder, recovery, generator, supervisor, discriminator])
print(f"Parámetros totales: {total_params:,}")
for name, m in [("Embedder", embedder), ("Recovery", recovery), ("Generator", generator),
                ("Supervisor", supervisor), ("Discriminator", discriminator)]:
    print(f"  {name:<15} {count_params(m):>8,}")

# DataLoader
ts_tensor = torch.from_numpy(ts)
dataset   = TensorDataset(ts_tensor)
loader    = DataLoader(dataset, batch_size=BATCH, shuffle=True, drop_last=True, num_workers=0)

## 3. Fase 1 — Pre-entrenamiento del autoencoder (Embedder + Recovery)

In [ ]:
opt_er = torch.optim.Adam(
    list(embedder.parameters()) + list(recovery.parameters()), lr=1e-3
)

losses_ae = []
t0 = time.time()

for epoch in tqdm(range(1, PRETRAIN_E + 1), desc="Fase 1 — Autoencoder"):
    epoch_loss = 0.0
    for (x_batch,) in loader:
        x_batch = x_batch.to(DEVICE)
        h       = embedder(x_batch)
        x_hat   = recovery(h)
        loss    = reconstruction_loss(x_batch, x_hat)
        opt_er.zero_grad()
        loss.backward()
        opt_er.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(loader)
    losses_ae.append(avg)

print(f"Loss final autoencoder: {losses_ae[-1]:.5f}  ({(time.time()-t0)/60:.1f} min)")

## 4. Fase 2 — Pre-entrenamiento del Supervisor

In [ ]:
opt_s = torch.optim.Adam(supervisor.parameters(), lr=1e-3)

losses_sup = []
t0 = time.time()

embedder.eval()  # congelado durante el pre-entrenamiento del supervisor

for epoch in tqdm(range(1, PRETRAIN_S + 1), desc="Fase 2 — Supervisor"):
    epoch_loss = 0.0
    for (x_batch,) in loader:
        x_batch = x_batch.to(DEVICE)
        with torch.no_grad():
            h = embedder(x_batch)
        h_sup = supervisor(h)
        loss  = supervised_step_loss(h, h_sup)
        opt_s.zero_grad()
        loss.backward()
        opt_s.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(loader)
    losses_sup.append(avg)

embedder.train()
print(f"Loss final supervisor: {losses_sup[-1]:.5f}  ({(time.time()-t0)/60:.1f} min)")

## 5. Fase 3 — Entrenamiento conjunto adversarial

Orden de actualización por paso (siguiendo el paper):
1. Actualizar **Generator + Supervisor** (supervised loss sobre muestras sintéticas)
2. Actualizar **Generator + Supervisor** (adversarial loss)
3. Actualizar **Embedder + Recovery** (reconstruction + supervised loss)
4. Actualizar **Discriminator** solo si su loss > `D_THRESHOLD` (evita que domine)

In [ ]:
opt_g   = torch.optim.Adam(
    list(generator.parameters()) + list(supervisor.parameters()), lr=1e-3
)
opt_er2 = torch.optim.Adam(
    list(embedder.parameters()) + list(recovery.parameters()), lr=1e-3
)
opt_d   = torch.optim.Adam(discriminator.parameters(), lr=1e-3)

losses_joint = {"G": [], "D": [], "E": []}
t0 = time.time()

for epoch in tqdm(range(1, JOINT_EPOCHS + 1), desc="Fase 3 — Joint training"):
    g_losses, d_losses, e_losses = [], [], []

    for (x_batch,) in loader:
        x_batch = x_batch.to(DEVICE)
        B       = x_batch.size(0)
        z       = torch.rand(B, SEQ_LEN, NOISE_DIM, device=DEVICE)

        # ---- Embeddings reales ----
        h_real = embedder(x_batch)

        # ---- Paso 1 & 2: actualizar G + S ----
        opt_g.zero_grad()

        h_fake    = generator(z)
        h_sup_fake = supervisor(h_fake)
        x_hat_fake = recovery(h_sup_fake)

        # Supervised loss sobre muestras sintéticas
        l_sup  = supervised_step_loss(h_fake, h_sup_fake)
        # Adversarial loss
        d_fake = discriminator(h_sup_fake)
        l_adv  = generator_adversarial_loss(d_fake)
        # Moments loss
        l_mom  = moments_loss(x_batch, x_hat_fake)

        l_g = l_adv + GAMMA * l_mom + ETA * l_sup
        l_g.backward()
        torch.nn.utils.clip_grad_norm_(
            list(generator.parameters()) + list(supervisor.parameters()), 1.0
        )
        opt_g.step()
        g_losses.append(l_g.item())

        # ---- Paso 3: actualizar E + R ----
        opt_er2.zero_grad()

        h_real2  = embedder(x_batch)
        x_recon  = recovery(h_real2)
        h_sup_r  = supervisor(h_real2)

        l_recon = reconstruction_loss(x_batch, x_recon)
        l_sup_r = supervised_step_loss(h_real2, h_sup_r)
        l_e     = ETA * l_sup_r + l_recon

        l_e.backward()
        torch.nn.utils.clip_grad_norm_(
            list(embedder.parameters()) + list(recovery.parameters()), 1.0
        )
        opt_er2.step()
        e_losses.append(l_e.item())

        # ---- Paso 4: actualizar D (condicional) ----
        with torch.no_grad():
            h_fake_det = generator(z)
            h_sup_det  = supervisor(h_fake_det)
            h_real_det = embedder(x_batch)

        d_real = discriminator(h_real_det)
        d_fake = discriminator(h_sup_det)
        l_d    = discriminator_loss(d_real, d_fake)

        if l_d.item() > D_THRESHOLD:
            opt_d.zero_grad()
            l_d.backward()
            torch.nn.utils.clip_grad_norm_(discriminator.parameters(), 1.0)
            opt_d.step()
        d_losses.append(l_d.item())

    losses_joint["G"].append(np.mean(g_losses))
    losses_joint["D"].append(np.mean(d_losses))
    losses_joint["E"].append(np.mean(e_losses))

    if epoch % 100 == 0:
        elapsed = (time.time() - t0) / 60
        tqdm.write(
            f"Epoch {epoch:>4}/{JOINT_EPOCHS}  "
            f"G: {losses_joint['G'][-1]:.4f}  "
            f"D: {losses_joint['D'][-1]:.4f}  "
            f"E: {losses_joint['E'][-1]:.4f}  "
            f"({elapsed:.1f} min)"
        )

total_time = (time.time() - t0) / 60
print(f"\nEntrenamiento conjunto completado en {total_time:.1f} min.")

# Guardar checkpoint
torch.save(
    {
        "embedder":      embedder.state_dict(),
        "recovery":      recovery.state_dict(),
        "generator":     generator.state_dict(),
        "supervisor":    supervisor.state_dict(),
        "discriminator": discriminator.state_dict(),
        "losses_ae":     losses_ae,
        "losses_sup":    losses_sup,
        "losses_joint":  losses_joint,
        "hyperparams":   {
            "hidden_dim": HIDDEN_DIM, "num_layers": NUM_LAYERS,
            "noise_dim": NOISE_DIM, "seq_len": SEQ_LEN, "n_features": N_FEATURES,
        },
    },
    MODELS / "timegan_checkpoint.pt",
)
print("Checkpoint guardado: models/timegan_checkpoint.pt")

In [ ]:
# Curvas de convergencia — las 3 fases
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(losses_ae, color="steelblue", linewidth=1.0)
axes[0].set_title("Fase 1 — Autoencoder (Reconstruction MSE)")
axes[0].set_xlabel("Época")

axes[1].plot(losses_sup, color="seagreen", linewidth=1.0)
axes[1].set_title("Fase 2 — Supervisor (Step MSE)")
axes[1].set_xlabel("Época")

for key, color in [("G", "tomato"), ("D", "steelblue"), ("E", "seagreen")]:
    axes[2].plot(losses_joint[key], color=color, linewidth=0.8, label=key, alpha=0.8)
axes[2].set_title("Fase 3 — Joint training")
axes[2].set_xlabel("Época")
axes[2].legend()

plt.tight_layout()
plt.savefig(REPORTS / "timegan_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: reports/timegan_loss_curves.png")

## 6. Generación de series temporales sintéticas

In [ ]:
N_SAMPLES = len(ts)

generator.eval()
supervisor.eval()
recovery.eval()

synth_ts_list = []
with torch.no_grad():
    for start in tqdm(range(0, N_SAMPLES, BATCH), desc="Generando"):
        end   = min(start + BATCH, N_SAMPLES)
        b     = end - start
        z     = torch.rand(b, SEQ_LEN, NOISE_DIM, device=DEVICE)
        h_hat = generator(z)
        h_sup = supervisor(h_hat)
        x_hat = recovery(h_sup)
        synth_ts_list.append(x_hat.cpu().numpy())

synth_ts = np.concatenate(synth_ts_list, axis=0)
synth_ts = np.clip(synth_ts, 0.0, 1.0)  # Recovery usa Sigmoid pero clip por seguridad

np.save(SYNTHETIC / "timegan_samples.npy", synth_ts)
print(f"Guardado: data/synthetic/timegan_samples.npy  {synth_ts.shape}")
print(f"Rango generado: [{synth_ts.min():.4f}, {synth_ts.max():.4f}]")

## 7. Validación de las series generadas

### 7.1 Evolución temporal media — real vs sintético

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=False)
axes = axes.flatten()

for i, v in enumerate(VITAL_NAMES):
    ax = axes[i]
    real_mean = ts[:, :, i].mean(axis=0)
    real_std  = ts[:, :, i].std(axis=0)
    synt_mean = synth_ts[:, :, i].mean(axis=0)
    synt_std  = synth_ts[:, :, i].std(axis=0)

    ax.plot(real_mean, color="steelblue", linewidth=1.5, label="Real")
    ax.fill_between(range(SEQ_LEN), real_mean - real_std, real_mean + real_std,
                    color="steelblue", alpha=0.15)
    ax.plot(synt_mean, color="tomato", linewidth=1.5, label="TimeGAN", linestyle="--")
    ax.fill_between(range(SEQ_LEN), synt_mean - synt_std, synt_mean + synt_std,
                    color="tomato", alpha=0.15)
    ax.set_title(v, fontsize=9)
    ax.set_xlabel("Hora", fontsize=7)

axes[0].legend(fontsize=7)
for j in range(len(VITAL_NAMES), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Evolución temporal media (norm.) — Real vs TimeGAN", fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS / "timegan_temporal_evolution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: reports/timegan_temporal_evolution.png")

### 7.2 Distribuciones marginales por vital

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, v in enumerate(VITAL_NAMES):
    ax = axes[i]
    ax.hist(ts[:, :, i].flatten(),       bins=60, alpha=0.5, density=True,
            color="steelblue", label="Real")
    ax.hist(synth_ts[:, :, i].flatten(), bins=60, alpha=0.5, density=True,
            color="tomato", label="TimeGAN")
    ax.set_title(v, fontsize=9)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=7)
for j in range(len(VITAL_NAMES), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribuciones marginales — Real vs TimeGAN", fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS / "timegan_marginals.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: reports/timegan_marginals.png")

### 7.3 Análisis de GCS — masa en cero

GCS verbal y eye tienen alta proporción de ceros (pacientes intubados/sedados).
Se verifica si TimeGAN reproduce esta característica clínica.

In [ ]:
ZERO_THRESHOLD = 0.05  # consideramos "cero" cualquier valor < 0.05 (normalizado)

print(f"Análisis de masa en cero (umbral = {ZERO_THRESHOLD}):")
print(f"{'Vital':<15} {'Real %':>10} {'Sintético %':>12} {'Diferencia':>12}")
print("-" * 52)
gcs_issues = []
for i, v in enumerate(VITAL_NAMES):
    real_pct  = (ts[:, :, i] < ZERO_THRESHOLD).mean() * 100
    synth_pct = (synth_ts[:, :, i] < ZERO_THRESHOLD).mean() * 100
    diff      = synth_pct - real_pct
    flag      = "  ← revisar" if abs(diff) > 10 else ""
    print(f"{v:<15} {real_pct:>10.1f} {synth_pct:>12.1f} {diff:>+12.1f}{flag}")
    if abs(diff) > 10:
        gcs_issues.append(v)

print()
if gcs_issues:
    print(f"Vitales con discrepancia > 10pp en masa cero: {gcs_issues}")
    print("Nota: esta discrepancia es conocida para GAN con distribuciones bimodales.")
    print("Se documentará en la evaluación de fidelidad (Fase 3).")
else:
    print("Masa en cero reproducida correctamente en todos los vitales.")

## 8. Resumen

In [ ]:
print("=" * 60)
print("  RESUMEN — Notebook 06")
print("=" * 60)
checks = [
    ("Dataset real",                f"{ts.shape}"),
    ("hidden_dim / num_layers",     f"{HIDDEN_DIM} / {NUM_LAYERS}"),
    ("Parámetros totales",          f"{total_params:,}"),
    ("Loss AE final",               f"{losses_ae[-1]:.5f}"),
    ("Loss Supervisor final",       f"{losses_sup[-1]:.5f}"),
    ("Loss G final (joint)",        f"{losses_joint['G'][-1]:.4f}"),
    ("Loss D final (joint)",        f"{losses_joint['D'][-1]:.4f}"),
    ("Tiempo entrenamiento",        f"{total_time:.1f} min"),
    ("Muestras generadas",          f"{synth_ts.shape}"),
    ("Rango sintético",             f"[{synth_ts.min():.3f}, {synth_ts.max():.3f}]"),
]
for name, val in checks:
    print(f"  {name:<32} {val}")
print("=" * 60)
print("Listos para notebook 07 (DP-CTGAN con Opacus).")